# Part C (AM) — Interview Ready
**Day 33 | AM Session | Week 6**

Three interview-style questions: conceptual distinction, scratch implementation, and debugging.

---
## Q1 — Conceptual: SVM vs Logistic Regression

**Question:** SVM and Logistic Regression both find linear decision boundaries (with linear kernel). What is the fundamental difference in how they find the boundary? When would you prefer one over the other?

### Answer

**How each finds the boundary:**

| | Logistic Regression | SVM (linear kernel) |
|--|--|--|
| **Objective** | Minimises log-loss (cross-entropy) over *all* training points | Maximises the *margin* (gap) between classes; only **support vectors** matter |
| **Decision rule** | Probabilistic — boundary placed where P(y=1) = 0.5 | Geometric — boundary equidistant from the nearest points of each class |
| **Sensitivity** | Every point contributes to the gradient | Only support vectors (a small subset) influence the hyperplane |
| **Loss function** | Log-loss (penalises misclassification smoothly) | Hinge loss (zero for correctly classified points outside margin) |
| **Output** | Calibrated probability P(y|x) | Signed distance to hyperplane (no probability by default) |

**When to prefer LR:**
- You need probability estimates (e.g., risk scores, ranking).
- Dataset is large (>100K rows) — LR trains faster.
- Features are already well-scaled and roughly linearly separable.
- Interpretability matters — LR coefficients are directly interpretable as log-odds.

**When to prefer SVM:**
- Data is high-dimensional (e.g., text with TF-IDF, gene expression).
- Clear margin separability (or near-separable with soft margin C).
- Non-linear boundary needed — switch to RBF/poly kernel without changing the algorithm.
- Fewer samples relative to features — SVM generalises well via maximum margin.

> **One-line rule:** Use LR when you need *probabilities* or fast training on large data; use SVM when you need *margin-based robustness* or want to leverage kernel tricks for non-linear boundaries.

---
## Q2 — Coding: KNN from Scratch (NumPy only)

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


def knn_from_scratch(X_train: np.ndarray,
                     y_train: np.ndarray,
                     X_test: np.ndarray,
                     k: int) -> np.ndarray:
    """
    KNN classifier implemented with NumPy only.

    Parameters
    ----------
    X_train : (n_train, n_features)  training features
    y_train : (n_train,)             training labels
    X_test  : (n_test, n_features)   test features
    k       : int                    number of neighbours

    Returns
    -------
    predictions : (n_test,)  predicted class for each test sample
    """
    predictions = []

    for test_point in X_test:
        # Euclidean distance: sqrt(sum((x_train - x_test)^2))
        diffs     = X_train - test_point          # (n_train, n_features)
        sq_dists  = np.sum(diffs ** 2, axis=1)    # (n_train,)
        distances = np.sqrt(sq_dists)             # (n_train,)

        # Indices of k smallest distances
        k_indices = np.argpartition(distances, k)[:k]

        # Majority vote
        k_labels = y_train[k_indices]
        pred = np.bincount(k_labels).argmax()
        predictions.append(pred)

    return np.array(predictions)


# ----- Validation on digits dataset -----
digits = load_digits()
X, y   = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Use a small subset for speed (scratch implementation is O(n*d) per query)
y_pred_scratch = knn_from_scratch(X_train_sc, y_train, X_test_sc, k=3)
acc = accuracy_score(y_test, y_pred_scratch)
print(f'KNN-from-scratch accuracy (K=3): {acc:.4f}')

# Sanity-check against sklearn
from sklearn.neighbors import KNeighborsClassifier
sk_pred = KNeighborsClassifier(n_neighbors=3).fit(X_train_sc, y_train).predict(X_test_sc)
print(f'sklearn KNN accuracy            : {accuracy_score(y_test, sk_pred):.4f}')
print(f'Predictions match               : {np.array_equal(y_pred_scratch, sk_pred)}')

---
## Q3 — Debug: SVM with 0.50 Accuracy

```python
svm = SVC(kernel='rbf', C=1.0)
svm.fit(X_train, y_train)  # Features: salary (50K-200K), age (20-60)
print(svm.score(X_test, y_test))  # 0.50 = random!
```

### Root Cause

**Missing feature scaling.** The two features have wildly different scales:
- `salary`: 50 000 – 200 000
- `age`: 20 – 60

The RBF kernel computes `exp(-gamma * ||x_i - x_j||²)`. With unscaled data, `salary` dominates the Euclidean distance completely — the `age` feature becomes invisible. The SVM effectively sees every point as equally distant and can't form a meaningful boundary, resulting in ~50% accuracy (no better than chance on a balanced binary problem).

### Fix

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Simulate salary/age-like unscaled data
rng = np.random.default_rng(42)
n = 500
salary = rng.uniform(50_000, 200_000, n)
age    = rng.uniform(20, 60, n)
y      = (salary / 200_000 + age / 60 + rng.normal(0, 0.3, n) > 1.0).astype(int)

X_raw = np.column_stack([salary, age])
X_train, X_test, y_train, y_test = train_test_split(X_raw, y, random_state=42)

# ❌ BROKEN — no scaling
svm_broken = SVC(kernel='rbf', C=1.0)
svm_broken.fit(X_train, y_train)
print(f'Without scaling: {svm_broken.score(X_test, y_test):.4f}')

# ✅ FIXED — scale first
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=1.0, gamma='scale'))
])
pipeline.fit(X_train, y_train)
print(f'With scaling   : {pipeline.score(X_test, y_test):.4f}')

### Additional debugging checks (always run these):
1. **Check class balance** — 0.50 on a balanced binary task = random. Is the dataset 50/50?
2. **Check gamma** — `gamma='scale'` auto-selects a sensible value; default `gamma='scale'` in modern sklearn, but older versions default to `1/n_features`.
3. **Check C** — Very small C may underfitting; combine with a grid search after scaling.
4. **Inspect support vectors** — `svm.n_support_` should be much less than training set size after proper tuning.